In [3]:
import pandas as pd
import re
import json
import os
import warnings
warnings.filterwarnings('ignore')

print("="*70)
print("ФИНАЛЬНЫЙ ПАРСЕР - JSON + CSV")
print("="*70)

def parse_and_extract(file_path, year_label):
    """
    Парсит файл и извлекает только нужные столбцы
    """
    print(f"\n📁 {os.path.basename(file_path)}")
    
    if not os.path.exists(file_path):
        print(f"  ❌ Файл не найден")
        return None
    
    xl = pd.ExcelFile(file_path)
    all_data = []
    
    for sheet_name in xl.sheet_names:
        print(f"  📄 Лист: {sheet_name}...", end=" ")
        
        try:
            # Читаем без заголовков
            df = pd.read_excel(file_path, sheet_name=sheet_name, header=None)
            
            # Ищем строку с заголовками (где есть "Дата, время")
            header_row = None
            for idx in range(min(100, len(df))):
                row_text = ' '.join(str(x).lower() for x in df.iloc[idx] if pd.notna(x))
                if 'дата, время' in row_text:
                    header_row = idx
                    break
            
            if header_row is None:
                # Ищем по ключевым словам
                for idx in range(min(100, len(df))):
                    row_text = ' '.join(str(x).lower() for x in df.iloc[idx] if pd.notna(x))
                    if 'дата' in row_text and 'место' in row_text:
                        header_row = idx
                        break
            
            if header_row is None:
                print("нет заголовков")
                continue
            
            # Получаем заголовки
            headers = []
            for col in range(df.shape[1]):
                val = df.iloc[header_row, col] if header_row < len(df) else None
                if pd.notna(val):
                    headers.append(str(val).strip())
                else:
                    headers.append(f"col_{col}")
            
            # Данные начинаются со следующей строки
            data_df = df.iloc[header_row + 1:].copy()
            data_df.columns = headers
            
            # Ищем нужные столбцы по названиям
            needed_cols = {
                'Дата, время': None,
                'Про долж.': None,
                'Подвижной состав': None,
                'Место отказа': None,
                'Отказавшее тех.средство (по КАС АНТ)': None,
                'Служба': None,
                'Струк. подразд.': None
            }
            
            # Ищем соответствия
            for col in data_df.columns:
                col_lower = str(col).lower()
                
                if 'дата' in col_lower and 'время' in col_lower:
                    needed_cols['Дата, время'] = col
                elif 'про долж' in col_lower:
                    needed_cols['Про долж.'] = col
                elif 'подвижной' in col_lower or 'состав' in col_lower:
                    needed_cols['Подвижной состав'] = col
                elif 'место отказа' in col_lower:
                    needed_cols['Место отказа'] = col
                elif 'тех.средство' in col_lower or 'кас' in col_lower:
                    needed_cols['Отказавшее тех.средство (по КАС АНТ)'] = col
                elif col_lower == 'служба':
                    needed_cols['Служба'] = col
                elif 'струк' in col_lower or 'подразд' in col_lower:
                    needed_cols['Струк. подразд.'] = col
            
            # Если не нашли по точным названиям, ищем по номерам
            if needed_cols['Дата, время'] is None:
                if len(data_df.columns) > 3:
                    needed_cols['Дата, время'] = data_df.columns[3]
            
            if needed_cols['Место отказа'] is None:
                if len(data_df.columns) > 6:
                    needed_cols['Место отказа'] = data_df.columns[6]
            
            if needed_cols['Струк. подразд.'] is None:
                if len(data_df.columns) > 9:
                    needed_cols['Струк. подразд.'] = data_df.columns[9]
            
            # Создаем результат
            result = pd.DataFrame()
            for standard, actual in needed_cols.items():
                if actual is not None and actual in data_df.columns:
                    result[standard] = data_df[actual]
            
            if len(result) == 0:
                print("нет нужных столбцов")
                continue
            
            # Фильтруем строки с датами
            if 'Дата, время' in result.columns:
                # Преобразуем в даты
                result['Дата_парс'] = pd.to_datetime(result['Дата, время'], errors='coerce', dayfirst=True)
                result = result[result['Дата_парс'].notna()]
                result = result.drop('Дата_парс', axis=1)
            
            if len(result) > 0:
                result['Год'] = year_label
                result['Лист'] = sheet_name
                all_data.append(result)
                print(f"✅ {len(result)} записей")
            else:
                print("нет записей с датами")
                
        except Exception as e:
            print(f"ошибка: {e}")
    
    if all_data:
        return pd.concat(all_data, ignore_index=True)
    return None

# Обрабатываем файлы
files = [
    ('ОТС_2022-23.xls', '2022-2023'),
    ('ОТС_2024-2025.xls', '2024-2025')
]

print("\n🔍 НАЧАЛО ПАРСИНГА...")
print("="*70)

all_data = []
for file_path, year_label in files:
    df = parse_and_extract(file_path, year_label)
    if df is not None and len(df) > 0:
        all_data.append(df)

if not all_data:
    print("\n❌ НЕ НАЙДЕНО НИ ОДНОЙ ЗАПИСИ!")
    exit()

# Объединяем все данные
final_df = pd.concat(all_data, ignore_index=True)

print("\n" + "="*70)
print("📊 ИТОГИ ПАРСИНГА")
print("="*70)
print(f"✅ ВСЕГО ЗАПИСЕЙ: {len(final_df)}")
print(f"\n📅 ПО ГОДАМ:")
for year, count in final_df['Год'].value_counts().items():
    print(f"   {year}: {count} записей")
print(f"\n📄 ПО ЛИСТАМ:")
for sheet, count in final_df['Лист'].value_counts().items():
    print(f"   {sheet}: {count} записей")

# Функция извлечения станции
def extract_station(location):
    if pd.isna(location):
        return None
    text = str(location).upper()
    # Убираем РЕГ-Х
    text = re.sub(r'РЕГ-\d+,?\s*', '', text)
    # Ищем станции
    stations = re.findall(r'[А-ЯЁ][А-ЯЁ\-]{2,}', text)
    return stations[0] if stations else None

# Извлекаем станции
final_df['Станция'] = final_df['Место отказа'].apply(extract_station)

# Показываем статистику по станциям
stations_found = final_df['Станция'].notna().sum()
print(f"\n📍 СТАНЦИИ:")
print(f"   Найдено станций: {stations_found} из {len(final_df)} ({stations_found/len(final_df)*100:.1f}%)")
print(f"   Уникальных станций: {final_df['Станция'].nunique()}")

print(f"\n🏆 ТОП-15 СТАНЦИЙ ПО ОТКАЗАМ:")
for station, count in final_df['Станция'].value_counts().head(15).items():
    print(f"   {station}: {count}")

# Проверяем наличие координат
if os.path.exists('stations.csv'):
    print(f"\n🗺️ ДОБАВЛЕНИЕ КООРДИНАТ...")
    
    coords = pd.read_csv('stations.csv', encoding='utf-8-sig')
    coord_dict = {}
    for _, row in coords.iterrows():
        name = str(row['Станция']).strip().upper()
        try:
            lat = float(row['Широта'])
            lng = float(row['Долгота'])
            coord_dict[name] = (lat, lng)
            coord_dict[name.replace('-', '')] = (lat, lng)
        except:
            pass
    
    final_df['Широта'] = final_df['Станция'].apply(lambda x: coord_dict.get(str(x).upper(), (None, None))[0])
    final_df['Долгота'] = final_df['Станция'].apply(lambda x: coord_dict.get(str(x).upper(), (None, None))[1])
    
    with_coords = final_df[final_df['Широта'].notna()]
    print(f"   С координатами: {len(with_coords)} записей")
    
    if len(with_coords) > 0:
        # СОХРАНЯЕМ В CSV
        csv_output = 'failures_map.csv'
        with_coords.to_csv(csv_output, index=False, encoding='utf-8-sig')
        print(f"\n✅ CSV файл: {csv_output}")
        
        # СОХРАНЯЕМ В JSON
        map_data = []
        for _, row in with_coords.iterrows():
            map_data.append({
                'station': str(row['Станция']),
                'lat': float(row['Широта']),
                'lng': float(row['Долгота']),
                'date': str(row['Дата, время']),
                'location': str(row['Место отказа']),
                'equipment': str(row['Отказавшее тех.средство (по КАС АНТ)']) if pd.notna(row['Отказавшее тех.средство (по КАС АНТ)']) else '',
                'unit': str(row['Струк. подразд.']) if pd.notna(row['Струк. подразд.']) else '',
                'year': str(row['Год'])
            })
        
        json_output = 'failures_map.json'
        with open(json_output, 'w', encoding='utf-8') as f:
            json.dump(map_data, f, ensure_ascii=False, indent=2)
        
        print(f"\n{'='*70}")
        print("✅ ГОТОВО! ФАЙЛЫ ДЛЯ КАРТЫ:")
        print(f"   1. {csv_output} - CSV формат")
        print(f"   2. {json_output} - JSON формат")
        print(f"{'='*70}")
        print(f"   Всего точек на карте: {len(map_data)}")
        print(f"   По годам:")
        for year, count in with_coords['Год'].value_counts().items():
            print(f"     {year}: {count} точек")
        print(f"\n   Топ-10 станций:")
        for station, count in with_coords['Станция'].value_counts().head(10).items():
            print(f"     {station}: {count}")
    else:
        print("\n❌ Нет записей с координатами")
else:
    print(f"\n{'='*70}")
    print("⚠️ НЕТ ФАЙЛА С КООРДИНАТАМИ (stations.csv)")
    print(f"{'='*70}")
    
    # Создаем список станций для заполнения
    stations_list = final_df['Станция'].dropna().unique()
    stations_list = sorted([s for s in stations_list if len(str(s)) > 2])
    
    template = pd.DataFrame({
        'Станция': stations_list,
        'Широта': '',
        'Долгота': ''
    })
    template.to_csv('stations_template.csv', index=False, encoding='utf-8-sig')
    
    print(f"\n📝 СОЗДАН ФАЙЛ: stations_template.csv")
    print(f"   Уникальных станций: {len(stations_list)}")
    print(f"\n   ДАЛЬНЕЙШИЕ ДЕЙСТВИЯ:")
    print(f"   1. Откройте stations_template.csv")
    print(f"   2. Заполните координаты (Широта, Долгота)")
    print(f"   3. Сохраните как stations.csv")
    print(f"   4. Запустите скрипт снова - получите failures_map.csv и failures_map.json")
    
    # Сохраняем все данные (без координат) в CSV для проверки
    final_df.to_csv('all_failures_data.csv', index=False, encoding='utf-8-sig')
    print(f"\n📁 Дополнительно сохранен файл для проверки: all_failures_data.csv")
    print(f"   В нем {len(final_df)} записей без координат")

print("\n" + "="*70)
print("🎉 ГОТОВО!")
print("="*70)

ФИНАЛЬНЫЙ ПАРСЕР - JSON + CSV

🔍 НАЧАЛО ПАРСИНГА...

📁 ОТС_2022-23.xls
  📄 Лист: 2022г... ✅ 1686 записей
  📄 Лист: 2023г... ✅ 1381 записей

📁 ОТС_2024-2025.xls
  📄 Лист: 2024г... ✅ 1117 записей
  📄 Лист: 2025г... ✅ 1066 записей

📊 ИТОГИ ПАРСИНГА
✅ ВСЕГО ЗАПИСЕЙ: 5250

📅 ПО ГОДАМ:
   2022-2023: 3067 записей
   2024-2025: 2183 записей

📄 ПО ЛИСТАМ:
   2022г: 1686 записей
   2023г: 1381 записей
   2024г: 1117 записей
   2025г: 1066 записей

📍 СТАНЦИИ:
   Найдено станций: 5226 из 5250 (99.5%)
   Уникальных станций: 283

🏆 ТОП-15 СТАНЦИЙ ПО ОТКАЗАМ:
   ИНСКАЯ: 551
   ВХОДНАЯ: 255
   АЛТАЙСКАЯ: 226
   МОСКОВКА: 154
   РЗД: 101
   ВАЛИХАНОВО: 78
   ИРТЫШСКОЕ: 74
   КАРАСУК: 73
   АРТЫШТА: 52
   КРАСНЫЙ: 47
   ТАЙГА: 47
   АЛАМБАЙ: 45
   КОРМИЛОВКА: 43
   ЗУБКОВО: 42
   ОБЬ: 42

🗺️ ДОБАВЛЕНИЕ КООРДИНАТ...
   С координатами: 3392 записей

✅ CSV файл: failures_map.csv

✅ ГОТОВО! ФАЙЛЫ ДЛЯ КАРТЫ:
   1. failures_map.csv - CSV формат
   2. failures_map.json - JSON формат
   Всего точек на карте: 33

In [4]:
import pandas as pd

print("="*60)
print("ОБЪЕДИНЕНИЕ ФАЙЛОВ - РЕЗУЛЬТАТ В CSV")
print("="*60)

# 1. Загружаем файлы
print("\n📁 Загрузка файлов...")

# Файл со станциями (справочник)
stations_df = pd.read_csv('stations.csv', encoding='utf-8-sig')
print(f"   stations.csv: {len(stations_df)} станций")

# Файл с отказами
failures_df = pd.read_csv('failures_map.csv', encoding='utf-8-sig')
print(f"   failures_map.csv: {len(failures_df)} записей")

# 2. Приводим названия станций к единому формату
stations_df['Станция_поиск'] = stations_df['Станция'].str.upper().str.strip()
failures_df['Станция_поиск'] = failures_df['Станция'].str.upper().str.strip()

# 3. Объединяем по названию станции
print("\n🔄 Объединение данных...")

# Берем из справочника нужные колонки
stations_info = stations_df[['Станция_поиск', 'ШЧ', 'Широта', 'Долгота', 'GeoPoint']]

# Объединяем
merged_df = failures_df.merge(stations_info, on='Станция_поиск', how='left')

# Удаляем временную колонку
merged_df = merged_df.drop('Станция_поиск', axis=1)

# 4. Статистика
matched = merged_df['ШЧ'].notna().sum()
total = len(merged_df)

print(f"\n📊 РЕЗУЛЬТАТ:")
print(f"   Найдено соответствий: {matched} из {total} ({matched/total*100:.1f}%)")

# Показываем ненайденные станции
if total - matched > 0:
    missing = merged_df[merged_df['ШЧ'].isna()]['Станция'].unique()
    print(f"\n⚠️ Станции без координат ({len(missing)} шт.):")
    for station in list(missing)[:20]:
        print(f"   - {station}")

# 5. Сохраняем CSV
output_file = 'failures_complete.csv'
merged_df.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"\n✅ ГОТОВО! Файл сохранен: {output_file}")
print(f"   Всего записей: {len(merged_df)}")
print(f"   Столбцы: {list(merged_df.columns)}")

print("\n" + "="*60)
print("ГОТОВО!")
print("="*60)

ОБЪЕДИНЕНИЕ ФАЙЛОВ - РЕЗУЛЬТАТ В CSV

📁 Загрузка файлов...
   stations.csv: 301 станций
   failures_map.csv: 3392 записей

🔄 Объединение данных...

📊 РЕЗУЛЬТАТ:
   Найдено соответствий: 3428 из 3428 (100.0%)

✅ ГОТОВО! Файл сохранен: failures_complete.csv
   Всего записей: 3428
   Столбцы: ['Дата, время', 'Про долж.', 'Подвижной состав', 'Место отказа', 'Отказавшее тех.средство (по КАС АНТ)', 'Служба', 'Струк. подразд.', 'Год', 'Лист', 'Станция', 'Широта_x', 'Долгота_x', 'ШЧ', 'Широта_y', 'Долгота_y', 'GeoPoint']

ГОТОВО!
